# Introduction to Generative Models

A generative model is a computer simulation or probabilistic model (it's the same thing in the end) that encodes explicit rules about how data is generated. Instead of analyzing data backwards from observations, you write down causal assumptions about how variables influence one another, then run the simulation to generate synthetic data. This notebook demonstrates how different hypotheses translate into distinct code structures and conditional dependencies.

In [ ]:
import numpy as np


# Base Class: ApplicationSimulator

The ApplicationSimulator class defines the shared blueprint for all simulation models. It initializes the number of schools and outlines the required execution pipeline: generating applications, generating decisions, and returning the collected dataset containing student applications, decisions, and genders.

In [2]:
class ApplicationSimulator:
    def __init__(self, n_schools: int):
        self.n_schools = n_schools
        print("Initialised with ", n_schools, "schools")
    def generate_applications(self, n_students: int):
        pass
    def generate_decisions(self):
        pass
    def generate_data(self, n_students: int):
        self.generate_applications(n_students)
        self.generate_decisions()
        return self.application, self.decision, self.gender


# Model 1: Completely Independent Application

In `CompletelyIndependentApplication`, all variables are independent. Students choose schools based on a shared global probability distribution, and schools accept students based on uniform acceptance probabilities. Gender is generated randomly with no effect on choices or outcomes. This serves as a baseline null model where no demographic biases or dependencies exist.

In [3]:
# The students apply to schools independently.
class CompletelyIndependentApplication(ApplicationSimulator):
    def __init__(self, n_schools: int):
        super().__init__(n_schools)
        self.application_probability = np.random.dirichlet(np.ones(self.n_schools))
        print("Student apply with these probabilities:", self.application_probability)
        self.acceptance_probability = np.random.uniform(size = self.n_schools)
        print("Each school accepts students these probabilities:", self.acceptance_probability)
    def generate_applications(self, n_students: int):
        self.n_students = n_students
        self.application = np.zeros(n_students)
        self.gender = np.zeros(n_students)
        for s in range(self.n_students):
            self.gender[s] = np.random.choice(2)
            self.application[s] = np.random.choice(self.n_schools, p=self.application_probability)

    def generate_decisions(self):
        n_students = len(self.application)
        self.decision = np.zeros(n_students)
        for s in range(n_students):
            school = int(self.application[s])
            p = self.acceptance_probability[school]
            print(school)
            d = np.random.choice(2, p = [1 - p, p])
            self.decision[s] = d


In [4]:
test = CompletelyIndependentApplication(3)
applications, decisions, gender = test.generate_data(10)
print(applications, decisions, gender)

Initialised with  3 schools
Student apply with these probabilities: [0.21481263 0.61317179 0.17201558]
Each school accepts students these probabilities: [0.55080971 0.4031976  0.48871391]
1
0
1
1
2
1
1
2
1
1
[1. 0. 1. 1. 2. 1. 1. 2. 1. 1.] [1. 1. 0. 0. 0. 0. 0. 1. 1. 1.] [0. 1. 0. 1. 1. 0. 1. 0. 0. 1.]


# Model 2: Gender Dependent Application

In `GenderDependentApplication`, student application choices depend on their gender. Male and female students draw their application probabilities from separate [Dirichlet distributions](https://en.wikipedia.org/wiki/Dirichlet_distribution), meaning different demographic groups prefer different schools. However, school acceptance probabilities remain independent of gender. This models a scenario where group disparities stem solely from applicant choices rather than institutional decisions.

In [5]:
# In this model, the student's choices depend on their gender.
# The students apply to schools independently of each other.
# The schools accept students independently of their gender.
class GenderDependentApplication(ApplicationSimulator):
    def __init__(self, n_schools: int):
        super().__init__(n_schools)
        self.application_probability_male = np.random.dirichlet(np.ones(self.n_schools))
        self.application_probability_female = np.random.dirichlet(np.ones(self.n_schools))
        print("Student apply with these probabilities:",
              self.application_probability_male,
              self.application_probability_female)
        self.acceptance_probability = np.random.uniform(size = self.n_schools)
        print("Each school accepts students these probabilities:", self.acceptance_probability)
    def generate_applications(self, n_students: int):
        self.gender = np.random.choice(2, size = n_students)
        self.application = np.zeros(n_students)
        for s in range(n_students):
            if (self.gender[s]==0):
                self.application[s] = np.random.choice(self.n_schools, p=self.application_probability_male)
            else:
                self.application[s] = np.random.choice(self.n_schools, p=self.application_probability_female)
    def generate_decisions(self):
        n_students = len(self.application)
        self.decision = np.zeros(n_students)
        print("Generating decisions")
        for s in range(n_students):
            school = int(self.application[s])
            p = self.acceptance_probability[school]
            d = np.random.choice(2, p = [1 - p, p])
            self.decision[s] = d
            #print("school", school, "acceptance", d)
test = GenderDependentApplication(3)
data = test.generate_data(10)
print("Student gender  : ", data[2])
print("Student choices  : ", data[0])
print("School acceptance: ", data[1])

Initialised with  3 schools
Student apply with these probabilities: [0.69516604 0.15935765 0.1454763 ] [0.39800529 0.45210731 0.1498874 ]
Each school accepts students these probabilities: [0.32594241 0.18806225 0.9036066 ]
Generating decisions
Student gender  :  [1 0 1 0 1 1 1 0 0 1]
Student choices  :  [1. 0. 0. 0. 1. 2. 0. 2. 0. 0.]
School acceptance:  [1. 1. 0. 1. 0. 1. 0. 1. 0. 0.]


# Model 3: Dependent Application

In `DependentApplication`, both application choices and school acceptance probabilities depend on gender. Separate acceptance probabilities are generated for male and female students at every school. This models a scenario where both applicant behavior and institutional selection mechanisms differ across demographics.

In [12]:
# In this model, everything is dependent
# The students apply to schools dependent on their gender.
# The schools have different probabilities of accepting female and male students.
class DependentApplication(ApplicationSimulator):
    def __init__(self, n_schools: int):
        super().__init__(n_schools)
        self.application_probability_male = np.random.dirichlet(np.ones(self.n_schools))
        self.application_probability_female = np.random.dirichlet(np.ones(self.n_schools))
        print("Student apply with these probabilities:",
              self.application_probability_male,
              self.application_probability_female)
        self.acceptance_probability_male= np.random.uniform(size = self.n_schools)
        self.acceptance_probability_female = np.random.uniform(size = self.n_schools)
        print("Each school accepts students these probabilities:", 
              self.acceptance_probability_male,
                self.acceptance_probability_female)

    def generate_applications(self, n_students: int):
        self.gender = np.random.choice(2, size = n_students)
        self.application = np.zeros(n_students)
        for s in range(n_students):
            if (self.gender[s]==0):
                self.application[s] = np.random.choice(self.n_schools, p=self.application_probability_male)
            else:
                self.application[s] = np.random.choice(self.n_schools, p=self.application_probability_female)
    def generate_decisions(self):
        n_students = len(self.application)
        self.decision = np.zeros(n_students)
        for s in range(n_students):
            school = int(self.application[s])
            if (self.gender[s]==0):
                p = self.acceptance_probability_male[school]
            else:
                p = self.acceptance_probability_female[school]
            print(school)
            d = np.random.choice(2, p = [1 - p, p])
            self.decision[s] = d
test = DependentApplication(3)
test.generate_data(10)

Initialised with  3 schools
Student apply with these probabilities: [0.13267815 0.29837279 0.56894906] [0.24925719 0.31895786 0.43178494]
Each school accepts students these probabilities: [0.27583281 0.03848937 0.13493931] [0.94023656 0.21257912 0.71462777]
2
2
0
1
2
2
2
0
2
1


(array([2., 2., 0., 1., 2., 2., 2., 0., 2., 1.]),
 array([0., 0., 0., 1., 1., 1., 1., 1., 0., 1.]),
 array([0, 0, 1, 1, 1, 0, 1, 1, 0, 1]))

# Model 4: Grade Dependent Application

In `GradeDependentApplication`, an intermediate variable is introduced. Students receive grades that depend on their gender, and school acceptance decisions depend on whether a student's grade exceeds the school's acceptance threshold. This models confounding, showing how outcome disparities can be driven by mediating factors rather than direct bias.

This is an issue, because if we do not know the grade, we might think that it is the schools that discriminate. However, it is the grade itself that seems to be the result of some type of discrimination.

In [7]:
# Here the students get a grade, dependent on the gender
# Then the outcome also depends on the grade
class GradeDependentApplication(ApplicationSimulator):
    def __init__(self, n_schools: int):
        super().__init__(n_schools)
        self.application_probability_male = np.random.dirichlet(np.ones(self.n_schools))
        self.application_probability_female = np.random.dirichlet(np.ones(self.n_schools))
        print("Student apply with these probabilities:",
              self.application_probability_male,
              self.application_probability_female)
        self.acceptance_threshold = np.random.uniform(size = self.n_schools)*4
        print("Each school accepts students these thresholds:", self.acceptance_threshold)
    def generate_applications(self, n_students: int):
        self.gender = np.random.choice(2, size = n_students)
        self.application = np.zeros(n_students)
        self.grade = np.zeros(n_students)
        for s in range(n_students):
            if (self.gender[s]==0):
                self.application[s] = np.random.choice(self.n_schools, p=self.application_probability_male)
                self.grade[s] = np.random.choice(5, p=[0.1, 0.2, 0.3, 0.2, 0.2])
            else:
                self.application[s] = np.random.choice(self.n_schools, p=self.application_probability_female)
                self.grade[s] = np.random.choice(5, p=[0.1, 0.1, 0.25, 0.3, 0.25])
        print("Grades:", self.grade)
        
    def generate_decisions(self):
        n_students = len(self.application)
        self.decision = np.zeros(n_students)
        for s in range(n_students):
            school = int(self.application[s])
            if (self.grade[s] > self.acceptance_threshold[school]):
                self.decision[s] = 1
            else:
                self.decision[s] = 0
                
test = GradeDependentApplication(3)
test.generate_data(10)

Initialised with  3 schools
Student apply with these probabilities: [0.42991169 0.23322878 0.33685953] [0.16664257 0.43521943 0.398138  ]
Each school accepts students these thresholds: [0.2723512  2.08669513 2.05318451]
Grades: [0. 2. 3. 2. 3. 2. 2. 2. 4. 4.]


(array([2., 2., 2., 2., 2., 1., 1., 0., 1., 1.]),
 array([0., 0., 1., 0., 1., 0., 0., 1., 1., 1.]),
 array([0, 1, 0, 0, 0, 0, 1, 1, 1, 1]))

# Simpson's Paradox Demonstration

The final section uses numerical arrays to demonstrate Simpson's paradox. Aggregate acceptance rates across multiple departments can reverse or distort trends observed within individual departments due to differing application rates. This illustrates why causal graphs and generative models are necessary to correctly interpret observational data.

In [13]:
import pandas as pd
# This are male and female acceptance rates for each school
# so that acceptance_rates[s] correspond to those of school s [normalised to 100]
acceptance_rates = np.array([[62, 82], [63, 68], [37, 34], [33, 35], [28, 24], [6, 7]])

In [17]:
# Similarly, we see the application rates of males versus females in different schools [normalised to 1]
application_rates = np.array([[0.3, 0.3, 0.2, 0.1, 0.05, 0.05], [0.05, 0.05, 0.1, 0.2, 0.3, 0.3]])


In [18]:
# We can now calculate the probability of being accepted if you are male versus female.
# We see that, even though females are ALWAYS more likely to get accepted at any given school, their average acceptance rate is lower.

print("M acceptance: ", np.dot(acceptance_rates[:,0],application_rates[0]))
print("F acceptance: ", np.dot(acceptance_rates[:,1],application_rates[1]))

M acceptance:  49.89999999999999
F acceptance:  27.200000000000003


In [11]:
application_rates

array([[0.3 , 0.3 , 0.2 , 0.1 , 0.05, 0.05],
       [0.05, 0.05, 0.1 , 0.2 , 0.3 , 0.3 ]])